In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

sns.set(style="whitegrid")

In [2]:
BASE_PATH = Path(r"E:\Thesis")

PAD_PATH = BASE_PATH / "PAD-UFES-20"
PAD_IMAGE_DIR = PAD_PATH / "images"
PAD_META = PAD_PATH / "metadata.csv"

MODEL_PATH = BASE_PATH / "outputs" / "efficientnetb3_baseline" / "best_effb3.pth"

OUT_DIR = BASE_PATH / "outputs" / "efficientnetb3_crossdomain_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (300, 300)
BATCH_SIZE = 4
CLASS_NAMES = ["ACK", "BCC", "MEL", "NEV"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

print("PAD meta exists:", PAD_META.exists())
print("Model exists:", MODEL_PATH.exists())
print("Output dir:", OUT_DIR)

PAD meta exists: True
Model exists: True
Output dir: E:\Thesis\outputs\efficientnetb3_crossdomain_experiments


In [3]:
pad = pd.read_csv(PAD_META)

pad_4 = pad[pad["diagnostic"].isin(CLASS_NAMES)].copy()
pad_4["label"] = pad_4["diagnostic"]
pad_4["image_path"] = pad_4["img_id"].apply(lambda x: str(PAD_IMAGE_DIR / x))
pad_4 = pad_4[pad_4["image_path"].apply(os.path.exists)].copy()

print("PAD samples:", len(pad_4))
print(pad_4["label"].value_counts())

PAD samples: 1871
label
BCC    845
ACK    730
NEV    244
MEL     52
Name: count, dtype: int64


In [4]:
def resolve_image_dir(folder: Path) -> Path:
    if (folder / "images").exists():
        return folder / "images"
    return folder

In [5]:
def build_df_from_folder(base_df: pd.DataFrame, folder: Path) -> pd.DataFrame:
    df = base_df.copy()
    df["image_path"] = df["img_id"].apply(lambda x: str(folder / x))
    df["exists"] = df["image_path"].apply(os.path.exists)
    df = df[df["exists"]].copy()
    return df

In [6]:
class PADDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.label_to_idx = CLASS_TO_IDX

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["image_path"]
        label = row["label"]

        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        y = self.label_to_idx[label]
        return image, torch.tensor(y, dtype=torch.long)

In [7]:
def make_loader(df: pd.DataFrame, shuffle: bool = False):
    transform = transforms.Compose([
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])

    dataset = PADDataset(df, transform=transform)
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    return loader

In [8]:
def compute_metrics(true_classes, pred_classes, pred_probs, class_names):
    acc = accuracy_score(true_classes, pred_classes)
    prec = precision_score(true_classes, pred_classes, average="macro", zero_division=0)
    rec = recall_score(true_classes, pred_classes, average="macro", zero_division=0)
    f1 = f1_score(true_classes, pred_classes, average="macro", zero_division=0)

    true_onehot = np.eye(len(class_names))[true_classes]
    auc = roc_auc_score(true_onehot, pred_probs, average="macro", multi_class="ovr")

    report = classification_report(
        true_classes,
        pred_classes,
        target_names=class_names,
        zero_division=0
    )

    cm = confusion_matrix(true_classes, pred_classes)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "report": report,
        "cm": cm,
    }

In [9]:
model = timm.create_model(
    "efficientnet_b3",
    pretrained=False,
    num_classes=len(CLASS_NAMES)
)

print(model.__class__.__name__)

EfficientNet


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = torch.load(MODEL_PATH, map_location=device)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Loaded model from model_state_dict.")
elif isinstance(checkpoint, dict):
    try:
        model.load_state_dict(checkpoint)
        print("Loaded model from raw state_dict.")
    except Exception as e:
        raise ValueError(f"Could not load checkpoint as state_dict: {e}")
else:
    raise ValueError("Unsupported checkpoint format.")

model = model.to(device)
model.eval()
print("EfficientNetB3 model loaded successfully.")

Loaded model from model_state_dict.
EfficientNetB3 model loaded successfully.


In [11]:
experiments = {
    "Original PAD": PAD_IMAGE_DIR,
    "Brightness Normalized PAD": BASE_PATH / "outputs" / "brightness_normalization_test" / "PAD_UFES_20_brightness_normalized_to_HAM",
    "Color Normalized PAD": BASE_PATH / "outputs" / "color_normalization_test" / "PAD_UFES_20_color_normalized_to_HAM",
    "Contrast Normalized PAD": BASE_PATH / "outputs" / "contrast_normalization_test" / "PAD_UFES_20_contrast_normalized_to_HAM",
    "Combined Normalized PAD": BASE_PATH / "outputs" / "combined_normalization_test" / "PAD_UFES_20_combined_normalized_to_HAM",
    "Histogram Matched PAD": BASE_PATH / "PAD_HIST_MATCH_IMPROVED",
    "Histogram + CLAHE PAD": BASE_PATH / "PAD_HIST_CLAHE_IMPROVED",
}

In [12]:
experiment_dfs = {}

for name, folder in experiments.items():
    folder = resolve_image_dir(folder)
    df = build_df_from_folder(pad_4, folder)
    experiment_dfs[name] = df
    print(f"{name}: {len(df)} images | folder: {folder}")

Original PAD: 1871 images | folder: E:\Thesis\PAD-UFES-20\images
Brightness Normalized PAD: 1871 images | folder: E:\Thesis\outputs\brightness_normalization_test\PAD_UFES_20_brightness_normalized_to_HAM
Color Normalized PAD: 1871 images | folder: E:\Thesis\outputs\color_normalization_test\PAD_UFES_20_color_normalized_to_HAM
Contrast Normalized PAD: 1871 images | folder: E:\Thesis\outputs\contrast_normalization_test\PAD_UFES_20_contrast_normalized_to_HAM
Combined Normalized PAD: 1871 images | folder: E:\Thesis\outputs\combined_normalization_test\PAD_UFES_20_combined_normalized_to_HAM
Histogram Matched PAD: 1871 images | folder: E:\Thesis\PAD_HIST_MATCH_IMPROVED
Histogram + CLAHE PAD: 1871 images | folder: E:\Thesis\PAD_HIST_CLAHE_IMPROVED\images


In [13]:
criterion = nn.CrossEntropyLoss()

@torch.inference_mode()
def evaluate_experiment(name: str, df: pd.DataFrame):
    loader = make_loader(df, shuffle=False)

    total_loss = 0.0
    total_samples = 0

    all_probs = []
    all_preds = []
    all_true = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item() * labels.size(0)
        total_samples += labels.size(0)

        all_probs.append(probs.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_true.append(labels.cpu().numpy())

    avg_loss = total_loss / total_samples
    all_probs = np.concatenate(all_probs, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)
    all_true = np.concatenate(all_true, axis=0)

    metrics = compute_metrics(all_true, all_preds, all_probs, CLASS_NAMES)
    metrics["loss"] = avg_loss
    metrics["true"] = all_true
    metrics["preds"] = all_preds
    metrics["probs"] = all_probs

    print(f"\n=== {name} ===")
    print("Loss :", avg_loss)
    print("Accuracy :", metrics["accuracy"])
    print("Precision:", metrics["precision"])
    print("Recall   :", metrics["recall"])
    print("F1 Score :", metrics["f1"])
    print("AUC      :", metrics["auc"])
    print(metrics["report"])

    return metrics

In [14]:
results = {}
confusions = {}

for name, df in experiment_dfs.items():
    if len(df) == 0:
        print(f"Skipping {name} (no images found).")
        continue

    metrics = evaluate_experiment(name, df)
    results[name] = metrics
    confusions[name] = metrics["cm"]


=== Original PAD ===
Loss : 2.6487899563135406
Accuracy : 0.3548904329235703
Precision: 0.3188733828639057
Recall   : 0.36513465093009567
F1 Score : 0.26071405485280685
AUC      : 0.6234181146449352
              precision    recall  f1-score   support

         ACK       0.41      0.56      0.47       730
         BCC       0.56      0.07      0.13       845
         MEL       0.03      0.04      0.03        52
         NEV       0.27      0.79      0.41       244

    accuracy                           0.35      1871
   macro avg       0.32      0.37      0.26      1871
weighted avg       0.45      0.35      0.30      1871


=== Brightness Normalized PAD ===
Loss : 2.4143504603924693
Accuracy : 0.4035275253874933
Precision: 0.3604488913193107
Recall   : 0.38532106205908656
F1 Score : 0.30135676173470144
AUC      : 0.6455384285975148
              precision    recall  f1-score   support

         ACK       0.42      0.68      0.52       730
         BCC       0.62      0.10      0.17